In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import random
from torchvision import transforms
from PIL import Image
import timm 
import ast
import importlib.util

DSL_PATH = 'dsl.py'  # Update this path as needed
CONSTANTS_PATH = 'constants.py'  # Update this path as needed

# Load the dsl module dynamically
spec = importlib.util.spec_from_file_location("dsl_module", DSL_PATH)
dsl_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dsl_module)

# Load the constants module dynamically
spec_const = importlib.util.spec_from_file_location("constants_module", CONSTANTS_PATH)
constants_module = importlib.util.module_from_spec(spec_const)
spec_const.loader.exec_module(constants_module)

def parse_dsl_functions(dsl_path):
    """Parses the dsl.py file to extract functions with their argument and return types."""
    with open(dsl_path, 'r') as file:
        node = ast.parse(file.read(), filename=dsl_path)

    functions = []
    for n in node.body:
        if isinstance(n, ast.FunctionDef):
            func_name = n.name
            args = []
            # Extract argument names and types
            for arg in n.args.args:
                arg_name = arg.arg
                arg_type = None
                if arg.annotation:
                    arg_type = ast.unparse(arg.annotation)
                args.append((arg_name, arg_type))

            # Extract return type
            return_type = None
            if n.returns:
                return_type = ast.unparse(n.returns)

            # Get the function object from the module
            func = getattr(dsl_module, func_name, None)
            if func is not None:
                functions.append({
                    'name': func_name,
                    'function': func,
                    'args': args,
                    'return_type': return_type
                })
    return functions

dsl_functions = parse_dsl_functions(DSL_PATH)

# Constants from constants.py
constants_dict = {name: getattr(constants_module, name) for name in dir(constants_module) if not name.startswith('__')}

# Map types to possible constants
type_to_constants = {
    'Boolean': [True, False],
    'Integer': [v for v in constants_dict.values() if isinstance(v, int)],
    'IntegerTuple': [v for v in constants_dict.values() if isinstance(v, tuple) and all(isinstance(i, int) for i in v)],
    'Grid': [],  # Grids will be generated during sample generation
    'Object': [],  # Objects will be generated
    'Piece': [],  # Add other types as needed
}

def generate_sample():
    # Generate a random PRE grid (30x30)
    PRE = np.random.randint(0, 11, size=(30, 30), dtype=int)
    variables = {'PRE': PRE}
    variable_types = {'PRE': 'Grid'}

    grid = PRE.copy()

    # Randomly choose n from 1 to 50 with probability n / ((50*51)/2)
    n_values = np.arange(1, 51)
    probabilities = n_values / ((50 * 51) / 2)
    probabilities /= probabilities.sum()  # Normalize
    n = np.random.choice(n_values, p=probabilities)

    # Apply n functions
    for i in range(n):
        # Randomly select a function from dsl_functions
        func_info = random.choice(dsl_functions)
        func_name = func_info['name']
        func = func_info['function']
        arg_info = func_info['args']
        return_type = func_info['return_type']

        # Prepare arguments
        args = []
        for arg_name, arg_type in arg_info:
            # Skip 'self' or any arguments without a type
            if arg_type is None or arg_name == 'self':
                continue

            # Remove possible typing qualifiers (e.g., 'Optional', 'Union')
            arg_type_clean = arg_type.replace('Optional[', '').replace(']', '').replace('Union[', '').split(',')[0]

            # Select a value for the argument
            arg_value = None

            # First, check if we have variables of the required type
            matching_vars = [var_name for var_name, var_type in variable_types.items() if var_type == arg_type_clean]

            if matching_vars:
                # Randomly select one of the matching variables
                arg_value = variables[random.choice(matching_vars)]
            elif arg_type_clean in type_to_constants:
                # Randomly select a constant of the required type
                possible_values = type_to_constants[arg_type_clean]
                if possible_values:
                    arg_value = random.choice(possible_values)
            else:
                # Could not find a matching variable or constant; skip this function
                arg_value = None

            if arg_value is None:
                break  # Cannot proceed with this function due to missing argument

            args.append(arg_value)
        else:
            # All arguments have been successfully assigned
            try:
                # Apply the function
                output = func(*args)

                # Update variables and their types
                var_name = f'var_{i}'
                variables[var_name] = output
                variable_types[var_name] = return_type

                # If the return type is 'Grid', update the grid
                if return_type == 'Grid':
                    grid = output
            except Exception as e:
                # Handle exceptions, possibly due to incorrect arguments
                continue  # Skip this function and try another one
        # If we couldn't assign all arguments, continue to the next function
        if len(args) < len([a for a in arg_info if a[1] is not None and a[0] != 'self']):
            continue

    # Ensure the grid is still 30x30
    if grid.shape != (30, 30):
        # Resize or crop to 30x30
        grid = grid[:30, :30]
        if grid.shape[0] < 30 or grid.shape[1] < 30:
            # Pad with zeros
            grid = np.pad(grid, ((0, 30 - grid.shape[0]), (0, 30 - grid.shape[1])), 'constant', constant_values=0)

    POST = grid

    # Concatenate PRE and POST to form a 30x60 grid
    sample_grid = np.concatenate((PRE, POST), axis=1)

    # Label is n
    label = n

    return sample_grid, label

In [7]:
generate_sample()

(array([[ 8,  8,  2, ...,  0,  2,  6],
        [ 9, 10,  5, ...,  4,  5,  4],
        [ 1,  2,  4, ..., 10,  0, 10],
        ...,
        [10,  4,  7, ...,  0,  0,  0],
        [ 5, 10, 10, ...,  0,  0,  0],
        [ 9,  9,  6, ...,  0,  0,  0]]),
 18)